# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedKroush/Flyrank-ML1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
import pandas as pd
import numpy as np
import os
import getpass

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score
)

# Get Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

# Connect to DuckDB
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

fact_daily = f"""
read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
"""

# Build March 2026 page-level dataset
labeled = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {fact_daily}
        WHERE report_date >= '2026-03-01'
          AND report_date < '2026-04-01'
    ),

    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 15 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last15,

            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 15 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev15,

            AVG(f.gsc_avg_position) AS avg_position_month,

            SUM(f.gsc_impressions) AS impressions_month,
            SUM(f.gsc_clicks) AS clicks_month

        FROM {fact_daily} f, bounds b

        WHERE f.report_date >= '2026-03-01'
          AND f.report_date < '2026-04-01'

        GROUP BY 1, 2

        HAVING imp_prev15 >= 50
    )

    SELECT *,
           (imp_last15 < 0.8 * imp_prev15)::INT AS is_declining
    FROM windowed
""").df()

print(f"Rows loaded: {len(labeled):,}")
print(f"Decline base rate: {labeled['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 94,559
Decline base rate: 0.373


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice and why

I use a Random Forest classifier because the task is to prioritize pages that show an observed search-performance decline. Random Forest can capture nonlinear relationships between impressions, clicks, and average search position while remaining relatively easy to inspect through feature importance. I compare it against a simple transparent rule so that the additional complexity has to demonstrate useful performance.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Method choice

feature_cols = [
    "impressions_month",
    "clicks_month",
    "avg_position_month"
]

target_col = "is_declining"

X = labeled[feature_cols].copy()
y = labeled[target_col].astype(int)

print("Features:")
print(feature_cols)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target base rate: {y.mean():.3f}")

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

print("\nModel: RandomForestClassifier")
print("Random seed: 42")

Features:
['impressions_month', 'clicks_month', 'avg_position_month']

Feature matrix shape: (94559, 3)
Target base rate: 0.373

Model: RandomForestClassifier
Random seed: 42


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

The initial evaluation uses a 75/25 stratified random split with a fixed random seed of 42. Stratification preserves the observed decline rate in both subsets, making the comparison stable across classes. This is an initial evaluation rather than a production validation because pages may share client and temporal structure; a grouped-by-client or time-aware split would be a stronger next test.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 75/25 stratified split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("=== SPLIT DESIGN ===")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

print(f"\nTraining decline rate: {y_train.mean():.3f}")
print(f"Test decline rate: {y_test.mean():.3f}")

print("\nSplit: 75/25 stratified random")
print("Random seed: 42")

=== SPLIT DESIGN ===
Training rows: 70,919
Test rows: 23,640

Training decline rate: 0.373
Test decline rate: 0.373

Split: 75/25 stratified random
Random seed: 42


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Train + compare vs baseline

The Random Forest is trained only on the training portion and evaluated on the held-out test portion. The transparent baseline uses the same test observations and flags pages with at least 250 monthly impressions and an average position worse than 20. Both approaches are therefore compared on the same observations and against the same observed-decline target.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Train Random Forest
model.fit(X_train, y_train)

# Random Forest predictions
rf_pred = model.predict(X_test)
rf_prob = model.predict_proba(X_test)[:, 1]

# Transparent baseline
baseline_pred = (
    (X_test["impressions_month"] >= 250) &
    (X_test["avg_position_month"] > 20)
).astype(int)

# Metrics
results = pd.DataFrame([
    {
        "approach": "Transparent baseline",
        "precision": precision_score(
            y_test, baseline_pred, zero_division=0
        ),
        "recall": recall_score(
            y_test, baseline_pred, zero_division=0
        ),
        "accuracy": accuracy_score(
            y_test, baseline_pred
        ),
        "roc_auc": np.nan
    },
    {
        "approach": "Random Forest",
        "precision": precision_score(
            y_test, rf_pred, zero_division=0
        ),
        "recall": recall_score(
            y_test, rf_pred, zero_division=0
        ),
        "accuracy": accuracy_score(
            y_test, rf_pred
        ),
        "roc_auc": roc_auc_score(
            y_test, rf_prob
        )
    }
])

print("=== MODEL VS BASELINE ===")
print(results.round(3).to_string(index=False))

print(f"\nTest-set base rate: {y_test.mean():.3f}")

=== MODEL VS BASELINE ===
            approach  precision  recall  accuracy  roc_auc
Transparent baseline      0.402   0.173     0.595      NaN
       Random Forest      0.472   0.338     0.612    0.595

Test-set base rate: 0.373


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The Random Forest is intentionally evaluated at its default 0.5 probability threshold, so its predictions are selective rather than comprehensive. False negatives are important because some declining pages will not enter the review queue, while false positives consume limited editorial attention. Feature importance is interpreted as model association rather than causal influence: the model can identify which observed fields it relies on without showing that changing those fields would cause recovery.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Error analysis and feature importance

error_analysis = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "predicted": rf_pred
})

false_positives = (
    (error_analysis["actual"] == 0) &
    (error_analysis["predicted"] == 1)
).sum()

false_negatives = (
    (error_analysis["actual"] == 1) &
    (error_analysis["predicted"] == 0)
).sum()

true_positives = (
    (error_analysis["actual"] == 1) &
    (error_analysis["predicted"] == 1)
).sum()

true_negatives = (
    (error_analysis["actual"] == 0) &
    (error_analysis["predicted"] == 0)
).sum()

print("=== ERROR ANALYSIS ===")
print(f"True positives:  {true_positives:,}")
print(f"True negatives:  {true_negatives:,}")
print(f"False positives: {false_positives:,}")
print(f"False negatives: {false_negatives:,}")

print("\n=== FEATURE IMPORTANCE ===")

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(importance.to_string(index=False))

print("\nInterpretation:")
print("Higher importance means the Random Forest relied more heavily on that")
print("feature for its predictions. It does not imply causation.")

=== ERROR ANALYSIS ===
True positives:  2,980
True negatives:  11,479
False positives: 3,340
False negatives: 5,841

=== FEATURE IMPORTANCE ===
           feature  importance
avg_position_month    0.510985
 impressions_month    0.430396
      clicks_month    0.058619

Interpretation:
Higher importance means the Random Forest relied more heavily on that
feature for its predictions. It does not imply causation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.